# Évaluation locale par l’indice SDR

In [11]:
# %% 
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import joblib
import os

from src.config import (
    MODELS_DIR,
    RANDOM_STATE,
    N_NEIGHBORS_SDR,
    N_BOOTSTRAP_SDR,
    LAMBDA_SDR,
    SHAP_BACKGROUND_SIZE
)

from src.sdrlocc import compute_SDRloc


# =========================
# LOAD DATA
# =========================

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()

# FORCE CLEAN NUMERIC + ALIGN SAFE
X_train = X_train.select_dtypes(include=[np.number]).astype(float).reset_index(drop=True)
X_test = X_test.select_dtypes(include=[np.number]).astype(float).reset_index(drop=True)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


Shapes:
X_train: (16512, 11)
X_test : (4128, 11)


# =========================
# LOAD MODELS
# =========================

In [ ]:
rf = joblib.load(os.path.join(MODELS_DIR, "random_forest.pkl"))
xgb_model = joblib.load(os.path.join(MODELS_DIR, "xgboost.pkl"))

print("Models loaded successfully")


Models loaded successfully


In [ ]:
# SAFE SAMPLING

np.random.seed(RANDOM_STATE)

sample_size = min(200, len(X_test))
X_test_sample = X_test.sample(n=sample_size, random_state=RANDOM_STATE).copy()

print("Sample size:", X_test_sample.shape)

Sample size: (200, 11)


In [ ]:
# SAFE POINT BUILDER

def build_point(row, columns):
    return (
        pd.DataFrame([row])
        .reindex(columns=columns, fill_value=0)
        .astype(float)
    )


# =========================
# SDRloc RUN FUNCTION
# =========================

In [ ]:
def run_sdrloc(model):

    results = []

    for idx, row in X_test_sample.iterrows():

        #  SAFE ALIGNMENT
        point = build_point(row, X_train.columns)

        sdr, A, S, K, kl = compute_SDRloc(
            model,
            X_train,
            y_train,
            point,
            k=N_NEIGHBORS_SDR,
            B=N_BOOTSTRAP_SDR,
            lam=LAMBDA_SDR,
            background_size=SHAP_BACKGROUND_SIZE,
            random_state=RANDOM_STATE
        )

        results.append({
            "index": idx,
            "SDRloc": sdr,
            "A_local": A,
            "S_stability": S,
            "K_div": K,
            "KL": kl
        })

    return pd.DataFrame(results)


# =========================
# XGBOOST
# =========================

In [ ]:
df_xgb = run_sdrloc(xgb_model)
print("XGB done")

XGB done


In [19]:
print("\n=== XGBoost ===")
print(df_xgb.describe())


=== XGBoost ===
             index      SDRloc     A_local  S_stability       K_div  \
count   200.000000  200.000000  200.000000   200.000000  200.000000   
mean   1883.225000    0.602547    0.908940     0.741793    0.113208   
std    1206.477044    0.172662    0.060357     0.215266    0.067557   
min       8.000000    0.043777    0.680618     0.049787    0.021679   
25%     861.250000    0.536739    0.870706     0.648386    0.063019   
50%    1867.500000    0.652199    0.924237     0.813897    0.093212   
75%    2867.250000    0.720560    0.953427     0.898036    0.149943   
max    4110.000000    0.853709    0.995546     0.962012    0.372879   

               KL  
count  200.000000  
mean     0.123274  
std      0.081013  
min      0.021917  
25%      0.065092  
50%      0.097846  
75%      0.162452  
max      0.466615  


# =========================
# RANDOM FOREST
# =========================

In [ ]:
df_rf = run_sdrloc(rf)
print("RF done")


RF done


In [ ]:
# STATS

print("\n=== Random Forest ===")
print(df_rf.describe())


=== Random Forest ===
             index      SDRloc     A_local  S_stability       K_div  \
count   200.000000  200.000000  200.000000   200.000000  200.000000   
mean   1883.225000    0.627212    0.889560     0.793036    0.122139   
std    1206.477044    0.139934    0.077407     0.176670    0.070570   
min       8.000000    0.036578    0.642794     0.049787    0.021512   
25%     861.250000    0.595198    0.844449     0.726156    0.071223   
50%    1867.500000    0.658225    0.914564     0.844017    0.109952   
75%    2867.250000    0.707826    0.944607     0.917006    0.159775   
max    4110.000000    0.823073    0.996656     0.980140    0.362053   

               KL  
count  200.000000  
mean     0.133713  
std      0.084763  
min      0.021747  
25%      0.073887  
50%      0.116481  
75%      0.174086  
max      0.449500  


In [ ]:
# STATS

print("\n=== Random Forest ===")
print(df_rf.describe())

print("\n=== XGBoost ===")
print(df_xgb.describe())


=== Random Forest ===
             index      SDRloc     A_local  S_stability       K_div  \
count   200.000000  200.000000  200.000000   200.000000  200.000000   
mean   1883.225000    0.627212    0.889560     0.793036    0.122139   
std    1206.477044    0.139934    0.077407     0.176670    0.070570   
min       8.000000    0.036578    0.642794     0.049787    0.021512   
25%     861.250000    0.595198    0.844449     0.726156    0.071223   
50%    1867.500000    0.658225    0.914564     0.844017    0.109952   
75%    2867.250000    0.707826    0.944607     0.917006    0.159775   
max    4110.000000    0.823073    0.996656     0.980140    0.362053   

               KL  
count  200.000000  
mean     0.133713  
std      0.084763  
min      0.021747  
25%      0.073887  
50%      0.116481  
75%      0.174086  
max      0.449500  

=== XGBoost ===
             index      SDRloc     A_local  S_stability       K_div  \
count   200.000000  200.000000  200.000000   200.000000  200.000000  

In [ ]:
# MERGE

X_test_sample = X_test_sample.copy()

X_test_sample["SDRloc_RF"] = df_rf["SDRloc"].values
X_test_sample["SDRloc_XGB"] = df_xgb["SDRloc"].values


In [24]:
# =========================
# SAVE
# =========================
os.makedirs("../data", exist_ok=True)

output_path = "../data/test_points_with_sdr.csv"
X_test_sample.to_csv(output_path, index=False)

print(f"Saved → {output_path}")

Saved → ../data/test_points_with_sdr.csv


# Analyse SDRloc (Stabilité, Diversité, Robustesse locale)

---

## Objectif

Calculer les métriques SDRloc (Stabilité, Diversité, Robustesse locale) pour évaluer la confiance locale des prédictions des modèles **Random Forest** et **XGBoost** sur un échantillon de points de test.

---

## 1. Chargement des données et modèles

Les données prétraitées :

- $X_{train}$ : 16512 observations, 11 features
- $X_{test}$ : 4128 observations
- $y_{train}$ : variable cible

Les modèles Random Forest et XGBoost sont chargés depuis des fichiers `.pkl`.

---

## 2. Échantillonnage des points de test

On sélectionne un échantillon aléatoire de $200$ points dans $X_{test}$.

La reproductibilité est assurée par une graine aléatoire fixée :

$RANDOM\_STATE$

---

## 3. Fonction `run_sdrloc`

Pour chaque point $x_i$, la fonction :

1. Crée un DataFrame avec les colonnes de $X_{train}$
2. Remplit les features manquantes par $0$
3. Appelle `compute_SDRloc`

Elle retourne :

- $sdr$ : score global SDRloc  
- $A_{local}$ : accuracy locale  
- $S_{stability}$ : stabilité des explications SHAP  
- $K_{div}$ : divergence KL  
- $KL$ : divergence complémentaire  

Paramètres utilisés :

- $N_{neighbors}$ : nombre de voisins
- $N_{bootstrap}$ : bootstrap samples
- $\lambda$ : poids stabilité / diversité
- $S_{SHAP}$ : taille du background SHAP

---

## 4. Évaluation des modèles

Les deux modèles sont évalués séparément :

- Random Forest → $df_{rf}$
- XGBoost → $df_{xgb}$

Chaque DataFrame contient les métriques SDRloc pour chaque point.

---

## 5. Résultats statistiques

| Modèle        | SDRloc | $A_{local}$ | $S_{stability}$ | $K_{div}$ | $KL$  |
|---------------|--------|-------------|------------------|-----------|------|
| Random Forest | 0.627  | 0.890       | 0.793            | 0.122     | 0.134 |
| XGBoost       | 0.603  | 0.909       | 0.742            | 0.113     | 0.123 |

---

## Interprétation

- $SDRloc \approx 0.6$ → confiance locale correcte
- $A_{local} > 0.89$ → cohérence forte avec les voisins
- $S_{stability} > 0.74$ → explications SHAP stables
- $K_{div}, KL$ faibles → homogénéité locale des explications

---

## Comparaison

- Random Forest :
  - meilleure stabilité
  - meilleur SDRloc moyen

- XGBoost :
  - meilleure précision locale ($A_{local}$)

---

## 6. Sauvegarde des résultats

Les colonnes suivantes sont ajoutées :

- $SDRloc_{RF}$
- $SDRloc_{XGB}$

Export final :

$../data/test\_points\_with\_sdr.csv$

---

## Conclusion

L’analyse SDRloc montre que les deux modèles sont globalement fiables localement.

- Random Forest : plus stable
- XGBoost : plus précis localement

Ces métriques permettent d’identifier les zones d’incertitude et d’améliorer la robustesse des modèles.